# Preparação do dashboard — JMA no lugar do INMET
O dashboard usa o **modelo principal** (INMET + previsão numérica), mas alimentado **só com dados do JMA GSM**, já que a API do INMET não está entregando dados recentes.

Este notebook roda **uma vez** e gera dois arquivos para o repositório do dashboard:
1. **`mapeamento_jma_inmet.json`** — conversão estatística (mapeamento de quantis) de cada variável do JMA para a escala da estação do INMET, ajustada em 2017–2023. Sem isso, a pressão entraria ~85 mB acima do que o modelo conhece (nível do mar × altitude da estação) e a chuva com o viés de "chuvisco" do JMA.
2. **`backtest.json`** — teste retroativo honesto em 2024 – jun/2026: quanto o modelo perde quando o INMET é substituído pelo JMA. Esses números aparecem no dashboard.

No fim, faz uma **previsão real para amanhã**, testando o caminho que o script diário vai usar.

**Arquivos para enviar:** `inmet_uberaba_diario_rf.csv`, `modelo_chuva_amanha.joblib` e `pipeline_chuva.py` (do pacote do dashboard).

In [ ]:
import os
faltando = [f for f in ("inmet_uberaba_diario_rf.csv", "modelo_chuva_amanha.joblib", "pipeline_chuva.py")
            if not os.path.exists(f)]
if faltando:
    print("Envie:", faltando)
    from google.colab import files
    files.upload()

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, brier_score_loss)

import pipeline_chuva as pc

SEED = 42
INICIO, FIM = "2017-05-19", "2026-06-30"
CORTE = pd.Timestamp("2023-12-31")
CACHE = "jma_historico_dashboard.csv"

df = pd.read_csv("inmet_uberaba_diario_rf.csv", parse_dates=["data"], index_col="data")
pacote = joblib.load("modelo_chuva_amanha.joblib")
print("scikit-learn:", sklearn.__version__, "| base INMET:", df.shape, "| features do modelo:", len(pacote["features"]))

## 1. Histórico do JMA com todas as variáveis necessárias
Além das 7 variáveis já baixadas antes, agora entram **direção do vento** e **radiação solar** (o modelo usa vento u/v e radiação do INMET). Leva alguns minutos; fica em cache.

In [ ]:
if os.path.exists(CACHE):
    h_hist = pd.read_csv(CACHE, parse_dates=["time"], index_col="time")
    print("Cache carregado:", h_hist.shape)
else:
    h_hist = pc.baixar_historico(INICIO, FIM, progresso=lambda it: tqdm(list(it), desc="Baixando JMA"))
    h_hist.to_csv(CACHE)
    print("Baixado:", h_hist.shape)
print("Variáveis:", list(h_hist.columns))

## 2. Mapeamento de quantis JMA → INMET (ajustado só no período de treino)

In [ ]:
jma_d = pc.jma_para_inmet_diario(h_hist)
mapa = pc.ajustar_mapeamento(jma_d, df, INICIO, CORTE)
print(f"Variáveis mapeadas: {len(mapa)} de {len(pc.VARS_INMET)}")
faltam = [v for v in pc.VARS_INMET if v not in mapa]
if faltam:
    print("Sem mapeamento (ficarão como NaN ou valor bruto):", faltam)

with open("mapeamento_jma_inmet.json", "w") as f:
    json.dump({"periodo_ajuste": [INICIO, str(CORTE.date())], "variaveis": mapa}, f)
print("Salvo: mapeamento_jma_inmet.json")

# Conferência no período de TESTE (fora do ajuste): médias INMET × JMA bruto × JMA mapeado
jma_map = pc.aplicar_mapeamento(jma_d, mapa)
teste = slice(CORTE + pd.Timedelta(days=1), FIM)
conf = pd.DataFrame({
    "INMET (real)": df.loc[teste, pc.VARS_INMET].mean(),
    "JMA bruto": jma_d.loc[teste, pc.VARS_INMET].mean(),
    "JMA mapeado": jma_map.loc[teste, pc.VARS_INMET].mean(),
    "correlação diária (mapeado × real)": [jma_map.loc[teste, v].corr(df.loc[teste, v]) for v in pc.VARS_INMET],
}).round(2)
conf

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, v in zip(axs, ["pressao_media_mb", "precip_mm", "umid_min_pct"]):
    bins = 40
    ax.hist(df.loc[teste, v].dropna(), bins=bins, alpha=.5, density=True, label="INMET real")
    ax.hist(jma_d.loc[teste, v].dropna(), bins=bins, alpha=.4, density=True, label="JMA bruto")
    ax.hist(jma_map.loc[teste, v].dropna(), bins=bins, histtype="step", lw=2, density=True, label="JMA mapeado")
    ax.set_title(v); ax.legend(fontsize=8)
    if v == "precip_mm":
        ax.set_yscale("log")
plt.suptitle("Distribuições no período de teste (2024–2026)"); plt.tight_layout(); plt.show()

## 3. Teste retroativo (2024 – jun/2026)
Para ser honesto, retreinamos o modelo principal **só até 2023** (mesmos hiperparâmetros do modelo salvo) e o avaliamos no teste de três formas:
- **INMET real** — referência (como no notebook anterior);
- **JMA mapeado** — o que o dashboard vai usar;
- **JMA bruto** — JMA direto, sem conversão (para mostrar por que o mapeamento é necessário).

As features da previsão para amanhã são as mesmas nos três casos.

In [ ]:
nwp = pc.features_nwp(h_hist).shift(-1)                   # linha D recebe a previsão de D+1
NWP_COLS = [c for c in pacote["features"] if c.startswith("nwp_")]
INMET_COLS = [c for c in pacote["features"] if not c.startswith("nwp_")]
ALVO = "alvo_chuva_amanha"

tab_real = df.join(nwp)
tab_map = pc.derivar_features(jma_map).join(nwp)
tab_bruto = pc.derivar_features(jma_d).join(nwp)

base = tab_real.dropna(subset=[ALVO, NWP_COLS[0]])
tr = base.loc[:CORTE]
y = df[ALVO]
dias_teste = base.loc[CORTE + pd.Timedelta(days=1):].index

params = {k: v for k, v in pacote["modelo"].get_params().items()}
rf = RandomForestClassifier(**params).fit(tr[pacote["features"]], tr[ALVO].astype(int))

def prob_oof(X, yy, n_splits=5):
    oof = pd.Series(np.nan, index=X.index)
    for i_tr, i_va in TimeSeriesSplit(n_splits=n_splits).split(X):
        m = RandomForestClassifier(**params).fit(X.iloc[i_tr], yy.iloc[i_tr])
        oof.iloc[i_va] = m.predict_proba(X.iloc[i_va])[:, 1]
    ok = oof.notna()
    return oof[ok].values, yy[ok].values

oof, y_oof = prob_oof(tr[pacote["features"]], tr[ALVO].astype(int))
cal = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof, y_oof)
limiar = pacote["limiar"]
print("Modelo de teste treinado até", tr.index.max().date(), "| limiar:", limiar)

In [ ]:
def metricas(yv, prob, pred):
    return {"AUC": roc_auc_score(yv, prob), "AP": average_precision_score(yv, prob),
            "Brier": brier_score_loss(yv, prob) if np.nanmax(prob) <= 1 else np.nan,
            "precisao": precision_score(yv, pred, zero_division=0),
            "recall": recall_score(yv, pred, zero_division=0),
            "F1": f1_score(yv, pred, zero_division=0)}

yte = y.loc[dias_teste].astype(int)
cenarios, probs = {}, {}
for nome, tab in [("INMET real", tab_real), ("JMA mapeado (dashboard)", tab_map), ("JMA bruto", tab_bruto)]:
    p = cal.predict(rf.predict_proba(tab.loc[dias_teste, pacote["features"]])[:, 1])
    probs[nome] = p
    cenarios[nome] = metricas(yte, p, (p >= limiar).astype(int))
pn = tab_real.loc[dias_teste, f"nwp_{pc.MODELO}_precip_mm"].values
cenarios["Previsão numérica pura (JMA)"] = metricas(yte, pn, (pn >= 1).astype(int))
cenarios["Climatologia"] = metricas(
    yte, dias_teste.month.map(tr[ALVO].groupby(tr.index.month).mean()).values,
    (dias_teste.month.map(tr[ALVO].groupby(tr.index.month).mean()).values >= .5).astype(int))

tabela = pd.DataFrame(cenarios).T.round(3)
tabela

**Como ler:** a diferença entre "INMET real" e "JMA mapeado" é o custo de não ter a estação em tempo real. Se "JMA mapeado" continuar acima da "previsão numérica pura", o modelo ainda agrega valor no dashboard.

In [ ]:
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(5, 4.5))
for nome in ("INMET real", "JMA mapeado (dashboard)"):
    fr, mp = calibration_curve(yte, probs[nome], n_bins=8, strategy="quantile")
    ax.plot(mp, fr, marker="o", label=nome)
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.legend()
ax.set_xlabel("Probabilidade prevista"); ax.set_ylabel("Frequência observada")
ax.set_title("Confiabilidade no teste"); plt.tight_layout(); plt.show()

## 4. Salvar o resultado do teste para o dashboard

In [ ]:
p_dash = probs["JMA mapeado (dashboard)"]
fr, mp = calibration_curve(yte, p_dash, n_bins=8, strategy="quantile")
serie = pd.DataFrame({
    "data_alvo": (dias_teste + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    "prob": np.round(p_dash, 3),
    "choveu": yte.values,
    "chuva_obs_mm": df["alvo_precip_amanha_mm"].reindex(dias_teste).round(1).values,
    "chuva_jma_mm": np.round(pn, 1),
})
backtest = {
    "periodo": [serie["data_alvo"].iloc[0], serie["data_alvo"].iloc[-1]],
    "n_dias": int(len(serie)), "limiar": limiar,
    "metricas": {k: {m: (None if pd.isna(v) else round(float(v), 3)) for m, v in d.items()}
                 for k, d in cenarios.items()},
    "confiabilidade": {"prob_prevista": np.round(mp, 3).tolist(), "freq_observada": np.round(fr, 3).tolist()},
    "serie": json.loads(serie.to_json(orient="records")),
}
with open("backtest.json", "w") as f:
    json.dump(backtest, f, ensure_ascii=False)
print("Salvo: backtest.json —", len(serie), "dias")

## 5. Ensaio da previsão real (o mesmo caminho do script diário)

In [ ]:
h_obs, h_prev = pc.baixar_operacional()
tabela_op = pc.montar_tabela(h_obs, h_prev, mapa)
hoje = pd.Timestamp.now(tz="America/Sao_Paulo").tz_localize(None).normalize()
prob, X = pc.prever(pacote, tabela_op, hoje)
print(f"Previsão para {(hoje + pd.Timedelta(days=1)).date()}: {prob:.0%} de chance de chuva (≥ 1 mm)")
print(f"Features ausentes: {int(X.isna().sum())} de {len(X)}")

In [ ]:
# Baixe os dois arquivos e coloque na pasta 'modelo/' do repositório do dashboard
from google.colab import files
files.download("mapeamento_jma_inmet.json")
files.download("backtest.json")